In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.manifold import TSNE

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.embeddings import embed_texts, load_model

RESUME_PATH = ROOT / 'data' / 'raw' / 'Resume' / 'Resume.csv'
JD_PATH = ROOT / 'data' / 'raw' / 'postings.csv'
assert RESUME_PATH.exists() and JD_PATH.exists(), 'Download the raw datasets first.'
RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 100)

In [ ]:
# Select exactly 100 items from each source. The sample intentionally contains Python-heavy resumes and one Python-developer JD.
resumes_all = pd.read_csv(RESUME_PATH, usecols=['ID', 'Category', 'Resume_str'])
jds_all = pd.read_csv(JD_PATH, usecols=['job_id', 'company_name', 'title', 'description'], low_memory=False)

python_resumes = resumes_all[resumes_all.Resume_str.fillna('').str.contains(r'\bpython\b', case=False, regex=True)]
python_jds = jds_all[jds_all.title.fillna('').str.contains(r'python.*developer|developer.*python', case=False, regex=True) | jds_all.description.fillna('').str.contains(r'python.*developer|developer.*python', case=False, regex=True)]
assert len(python_resumes) >= 5 and not python_jds.empty, 'Expected Python examples were not found.'

seed_resumes = python_resumes.sample(min(25, len(python_resumes)), random_state=RANDOM_STATE)
resumes = pd.concat([seed_resumes, resumes_all.drop(seed_resumes.index).sample(100 - len(seed_resumes), random_state=RANDOM_STATE)]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
python_jd = python_jds.sample(1, random_state=RANDOM_STATE)
jds = pd.concat([python_jd, jds_all.drop(python_jd.index).sample(99, random_state=RANDOM_STATE)]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

resumes['text'] = resumes.Resume_str.fillna('').str.replace(r'\s+', ' ', regex=True).str.strip().str[:6000]
jds['text'] = (jds.title.fillna('') + '. ' + jds.description.fillna('')).str.replace(r'\s+', ' ', regex=True).str.strip().str[:6000]
python_jd_id = python_jd.job_id.iloc[0]
print(f'Resumes: {resumes.shape}; JDs: {jds.shape}; chosen Python JD: {python_jd_id}')

In [ ]:
model = load_model()
resume_embeddings = embed_texts(resumes.text.tolist(), model, batch_size=32)
jd_embeddings = embed_texts(jds.text.tolist(), model, batch_size=32)

# Inputs are normalized by embed_texts, so dot product is cosine similarity.
similarity = resume_embeddings @ jd_embeddings.T
print(f'Resume vectors: {resume_embeddings.shape}; JD vectors: {jd_embeddings.shape}')
print(f'Pairwise similarity matrix: {similarity.shape}; range {similarity.min():.3f} to {similarity.max():.3f}')

In [ ]:
# Top-five pairs from all 10,000 resume/JD comparisons.
pair_positions = np.argsort(similarity.ravel())[-5:][::-1]
top_pairs = []
for position in pair_positions:
    resume_index, jd_index = np.unravel_index(position, similarity.shape)
    top_pairs.append({'cosine_similarity': round(float(similarity[resume_index, jd_index]), 4), 'resume_id': resumes.loc[resume_index, 'ID'], 'resume_category': resumes.loc[resume_index, 'Category'], 'jd_title': jds.loc[jd_index, 'title'], 'company': jds.loc[jd_index, 'company_name'], 'resume_preview': resumes.loc[resume_index, 'text'][:220]})
display(pd.DataFrame(top_pairs))

# Key check: the top resumes for the deliberately included Python-developer JD.
python_jd_index = jds.index[jds.job_id.eq(python_jd_id)][0]
python_scores = similarity[:, python_jd_index]
top_resume_indices = np.argsort(python_scores)[-5:][::-1]
print('Python developer JD:', jds.loc[python_jd_index, 'title'])
python_matches = pd.DataFrame({'cosine_similarity': python_scores[top_resume_indices].round(4), 'resume_id': resumes.loc[top_resume_indices, 'ID'].to_numpy(), 'category': resumes.loc[top_resume_indices, 'Category'].to_numpy(), 'mentions_python': resumes.loc[top_resume_indices, 'text'].str.contains(r'\bpython\b', case=False, regex=True).to_numpy(), 'resume_preview': resumes.loc[top_resume_indices, 'text'].str[:300].to_numpy()})
display(python_matches)
print(f"Python-mentioned resumes in top five: {python_matches.mentions_python.sum()}/5")

In [ ]:
# t-SNE: resumes are coloured by their known category; JDs are black crosses.
all_embeddings = np.vstack([resume_embeddings, jd_embeddings])
coordinates = TSNE(n_components=2, perplexity=min(30, (len(all_embeddings) - 1) // 3), init='pca', learning_rate='auto', random_state=RANDOM_STATE).fit_transform(all_embeddings)
plot_df = pd.DataFrame({'x': coordinates[:, 0], 'y': coordinates[:, 1], 'kind': ['Resume'] * len(resumes) + ['JD'] * len(jds), 'category': resumes.Category.fillna('Missing').tolist() + ['Job description'] * len(jds)})

plt.figure(figsize=(14, 10))
sns.scatterplot(data=plot_df[plot_df.kind.eq('Resume')], x='x', y='y', hue='category', palette='tab20', s=55, alpha=0.8)
sns.scatterplot(data=plot_df[plot_df.kind.eq('JD')], x='x', y='y', color='black', marker='x', s=38, alpha=0.4, label='JD')
plt.title('t-SNE of resume and JD embeddings')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Resume category')
plt.tight_layout()
plt.show()
print('Interpret qualitatively: nearby points should share role/skill language. Use the cosine rankings for retrieval decisions.')